<a href="https://colab.research.google.com/github/kerenG99/UFZ-Helmoltz-Summer-School-2026/blob/main/notebooks/09_creating_changing_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 09 · Bonus — creating & changing data

> **Trends in multi-omics data analysis for Microbial Ecology and Biotechnology** - Summer School, UFZ Leipzig  
> *Using SQLs for omics: basics and hands-on* - Instructor: Anderson Santos

**Estimated time: ~20 min**

## Learning objectives
- create a table with `CREATE TABLE` and add rows with `INSERT`;
- change rows with `UPDATE` and remove them with `DELETE`;
- save a query as a reusable `VIEW`.

So far we only read data. SQL can also create tables and change rows. These examples use a throwaway table called `my_notes`, so they never touch the study data.

---
| # | Lesson |
|---|---|
| 00 | Database foundations (concepts) |
| 01 | Meet the database & first SELECT |
| 02 | Filtering rows (WHERE) |
| 03 | Sorting, limiting & computed columns |
| 04 | Aggregation (GROUP BY) |
| 05 | Joining tables (JOIN) |
| 06 | Subqueries & CTEs |
| 07 | Complete & complex queries |
| 08 | Hands-on capstone |
| 09 | Bonus: creating & changing data |

## Setup — run this cell first

This connects the notebook to the example database. It works both on your own computer and on Google Colab; just run it (Shift+Enter). You do not need to understand it.

In [ ]:
# Run me first. Works locally AND on Google Colab.
%pip install jupysql --quiet
import os
REPO = "andersonavilasantos/ufz-summerschool-sql"   # course repository
if not os.path.exists("../data/omics.db"):
    # Not found locally -> on Colab: download the course and enter it.
    os.system(f"git clone -q https://github.com/{REPO}.git course_repo")
    if os.path.exists("course_repo/notebooks"):
        os.chdir("course_repo/notebooks")
%load_ext sql
%sql sqlite:///../data/omics.db
print("Connected to omics.db — you are ready.")

## 1. CREATE TABLE + INSERT

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS my_notes (
    id      INTEGER PRIMARY KEY,
    topic   TEXT,
    note    TEXT
);

In [ ]:
%%sql
INSERT INTO my_notes (id, topic, note) VALUES
    (1, 'joins', 'link tables on a shared key'),
    (2, 'groupby', 'one summary per group');

In [ ]:
%%sql
SELECT * FROM my_notes;

## 2. UPDATE — change existing rows

In [ ]:
%%sql
UPDATE my_notes SET note = 'the heart of relational SQL' WHERE topic = 'joins';

In [ ]:
%%sql
SELECT * FROM my_notes WHERE topic = 'joins';

## 3. DELETE — remove rows

In [ ]:
%%sql
DELETE FROM my_notes WHERE id = 2;

In [ ]:
%%sql
SELECT COUNT(*) AS remaining FROM my_notes;

## 4. VIEW — save a query under a name
A view behaves like a table but is really a stored query, handy for a summary you reuse often.

In [ ]:
%%sql
CREATE VIEW IF NOT EXISTS reads_per_environment AS
SELECT s.environment, SUM(a.count) AS reads
FROM abundance a
JOIN samples s ON a.sample_id = s.sample_id
GROUP BY s.environment;

In [ ]:
%%sql
SELECT * FROM reads_per_environment ORDER BY reads DESC;

## 5. Clean up
Drop the objects we created so the database is unchanged.

In [ ]:
%%sql
DROP VIEW IF EXISTS reads_per_environment;

In [ ]:
%%sql
DROP TABLE IF EXISTS my_notes;

---
> **Instructor note.** In real projects you rarely edit omics tables by hand, but `CREATE`/`INSERT` are how a pipeline loads data, and `VIEW`s are a clean way to share a standard summary query with colleagues.

### Recap
- `CREATE TABLE`, `INSERT`, `UPDATE`, `DELETE` create and change data.
- A `VIEW` stores a query under a name and is queried like a table.
- `IF NOT EXISTS` / `IF EXISTS` make scripts safe to re-run.
- With this, the SQL course is complete, and you can query a relational omics database on your own.